<a href="https://colab.research.google.com/github/trad024/Qdrant_essentials/blob/main/HNSW_Performance_Benchmarking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
import time
import numpy as np
import os
from google.colab import userdata
client = QdrantClient(url=userdata.get("QDRANT_URL"), api_key=userdata.get("QDRANT_API_KEY"))

encoder = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# Test configurations
configs = [
    {"name": "fast_initial_upload", "m": 0, "ef_construct": 100},  # m=0 = ingest-only
    {"name": "memory_optimized", "m": 8, "ef_construct": 100},  # m=8 = lower RAM
    {"name": "balanced", "m": 16, "ef_construct": 200},  # m=16 = balanced
    {"name": "high_quality", "m": 32, "ef_construct": 400},  # m=32 = higher recall, slower build
]

for config in configs:
    collection_name = f"my_domain_{config['name']}"
    if client.collection_exists(collection_name=collection_name):
        client.delete_collection(collection_name=collection_name)

    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
        hnsw_config=models.HnswConfigDiff(
            m=config["m"],
            ef_construct=config["ef_construct"],
            full_scan_threshold=10,  # force HNSW instead of full scan
        ),
        optimizers_config=models.OptimizersConfigDiff(
            indexing_threshold=10
        ),  # Force indexing even on small sets for demo
    )
    print(f"Created collection: {collection_name}")

Created collection: my_domain_fast_initial_upload
Created collection: my_domain_memory_optimized
Created collection: my_domain_balanced
Created collection: my_domain_high_quality


In [10]:
import time
import random
from qdrant_client import models

# ---------------------------------------------------
# Generate dataset (1000+ items with rich text field)
# ---------------------------------------------------
def generate_dataset(n=1200):
    topics = [
        "machine learning", "data engineering", "cloud computing",
        "vector databases", "natural language processing",
        "distributed systems", "search engines"
    ]

    dataset = []
    for i in range(n):
        topic = random.choice(topics)
        description = (
            f"This is a detailed description about {topic}. "
            f"It explains important ideas, key techniques, "
            f"and main applications used in production systems. "
            f"Document number {i} with additional contextual explanation."
        )

        dataset.append({
            "id": i,
            "description": description,
            "category": topic
        })

    return dataset


your_dataset = generate_dataset(1200)


# ---------------------------------------------------
# Upload with timing
# ---------------------------------------------------
def upload_with_timing(collection_name, data, config_name):
    print(f"\nEncoding embeddings for {config_name}...")
    embeddings = encoder.encode(
        [d["description"] for d in data],
        show_progress_bar=True
    ).tolist()

    points = []

    for i, item in enumerate(data):
        description = item.get("description", "")

        points.append(
            models.PointStruct(
                id=i,
                vector=embeddings[i],
                payload={
                    **item,
                    "length": len(description),                     # numeric field
                    "word_count": len(description.split()),         # numeric field
                    "has_keywords": any(                            # boolean field
                        keyword in description.lower()
                        for keyword in ["important", "key", "main"]
                    ),
                },
            )
        )

    print(f"Uploading {len(points)} points to {collection_name}...")

    start_time = time.time()
    client.upload_points(
        collection_name=collection_name,
        points=points,
    )
    upload_time = time.time() - start_time

    print(f"{config_name}: Uploaded {len(points)} points in {upload_time:.2f}s")

    # Warmup AFTER upload
    client.query_points(
        collection_name=collection_name,
        query=points[0].vector,
        limit=1,
    )

    return upload_time


# ---------------------------------------------------
# Upload to each collection
# ---------------------------------------------------
upload_times = {}

for config in configs:
    collection_name = f"my_domain_{config['name']}"
    upload_times[config["name"]] = upload_with_timing(
        collection_name,
        your_dataset,
        config["name"],
    )


# ---------------------------------------------------
# Wait for indexing
# ---------------------------------------------------
def wait_for_indexing(collection_name, timeout=120, poll_interval=2):
    print(f"\nWaiting for collection '{collection_name}' to be indexed...")
    start_time = time.time()

    while time.time() - start_time < timeout:
        info = client.get_collection(collection_name=collection_name)

        status = info.status

        # Handle different Qdrant client versions safely
        points_count = getattr(info, "points_count", None)
        indexed_vectors = getattr(info, "indexed_vectors_count", None)

        print(
            f" - Status: {status}, "
            f"Points: {points_count}, "
            f"Indexed: {indexed_vectors}"
        )

        if status == models.CollectionStatus.GREEN:
            print(f"✅ Collection '{collection_name}' is ready.")
            return

        time.sleep(poll_interval)

    raise TimeoutError(
        f"Timeout after {timeout}s. Collection '{collection_name}' not ready."
    )


for config in configs:
    if config["m"] > 0:  # Skip non-HNSW configs
        collection_name = f"my_domain_{config['name']}"
        wait_for_indexing(collection_name)


Encoding embeddings for fast_initial_upload...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Uploading 1200 points to my_domain_fast_initial_upload...
fast_initial_upload: Uploaded 1200 points in 3.44s

Encoding embeddings for memory_optimized...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Uploading 1200 points to my_domain_memory_optimized...
memory_optimized: Uploaded 1200 points in 3.65s

Encoding embeddings for balanced...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Uploading 1200 points to my_domain_balanced...
balanced: Uploaded 1200 points in 3.53s

Encoding embeddings for high_quality...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Uploading 1200 points to my_domain_high_quality...
high_quality: Uploaded 1200 points in 3.62s

Waiting for collection 'my_domain_memory_optimized' to be indexed...
 - Status: green, Points: 1200, Indexed: 1200
✅ Collection 'my_domain_memory_optimized' is ready.

Waiting for collection 'my_domain_balanced' to be indexed...
 - Status: green, Points: 1200, Indexed: 1200
✅ Collection 'my_domain_balanced' is ready.

Waiting for collection 'my_domain_high_quality' to be indexed...
 - Status: yellow, Points: 1200, Indexed: 1200
 - Status: green, Points: 1200, Indexed: 1200
✅ Collection 'my_domain_high_quality' is ready.


In [11]:
def benchmark_search(collection_name, query_embedding, ef_values=[64, 128, 256]):
    # Warmup
    client.query_points(collection_name=collection_name, query=query_embedding, limit=1)

    # hnsw_ef: higher = better recall, but slower. Tune per your latency goal.
    results = {}
    for hnsw_ef in ef_values:
        times = []

        # Run multiple queries for more reliable timing
        for _ in range(25):
            start_time = time.time()

            _ = client.query_points(
                collection_name=collection_name,
                query=query_embedding,
                limit=10,
                search_params=models.SearchParams(hnsw_ef=hnsw_ef),
                with_payload=False,
            )

            times.append((time.time() - start_time) * 1000)

        results[hnsw_ef] = {
            "avg_time": np.mean(times),
            "min_time": np.min(times),
            "max_time": np.max(times),
        }

    return results


test_query = "your test query"
query_embedding = encoder.encode(test_query).tolist()

performance_results = {}
for config in configs:
    if config["m"] > 0:  # Skip m=0 collections for search
        collection_name = f"my_domain_{config['name']}"
        performance_results[config["name"]] = benchmark_search(
            collection_name, query_embedding
        )

In [12]:
def test_filtering_performance(collection_name):
    query_embedding = encoder.encode("your filter test query").tolist()

    # Test filter without index
    filter_condition = models.Filter(
        must=[models.FieldCondition(key="length", range=models.Range(gte=10, lte=200))]
    )

    # Demo only: unindexed_filtering_retrieve=True forces a scan; turn it off right after measuring.
    client.update_collection(
        collection_name=collection_name,
        strict_mode_config=models.StrictModeConfig(unindexed_filtering_retrieve=True),
    )

    # Warmup
    client.query_points(collection_name=collection_name, query=query_embedding, limit=1)

    # Timing without payload index
    times = []
    for _ in range(25):
        start_time = time.time()
        _ = client.query_points(
            collection_name=collection_name,
            query=query_embedding,
            query_filter=filter_condition,
            limit=10,
            with_payload=False,
        )
        times.append((time.time() - start_time) * 1000)
    time_without_index = np.mean(times)

    # Create payload index
    client.create_payload_index(
        collection_name=collection_name,
        field_name="length",
        field_schema=models.PayloadSchemaType.INTEGER,
        wait=True,
    )

    # HNSW was already built; adding the payload index doesn’t rebuild it.
    # Bump ef_construct (+1) once to trigger a safe rebuild.
    base_ef = client.get_collection(
        collection_name=collection_name
    ).config.hnsw_config.ef_construct
    new_ef_construct = base_ef + 1

    client.update_collection(
        collection_name=collection_name,
        hnsw_config=models.HnswConfigDiff(ef_construct=new_ef_construct),
        strict_mode_config=models.StrictModeConfig(
            unindexed_filtering_retrieve=False
        ),  # Turn off scanning and use payload index instead.
    )

    wait_for_indexing(collection_name)

    # Warmup
    client.query_points(collection_name=collection_name, query=query_embedding, limit=1)

    # Timing with index
    times = []
    for _ in range(25):
        start_time = time.time()
        _ = client.query_points(
            collection_name=collection_name,
            query=query_embedding,
            query_filter=filter_condition,
            limit=10,
            with_payload=False,
        )
        times.append((time.time() - start_time) * 1000)
    time_with_index = np.mean(times)

    return {
        "without_index": time_without_index,
        "with_index": time_with_index,
        "speedup": time_without_index / time_with_index,
    }


# Test on your best performing collection
best_collection = "my_domain_balanced"  # Choose based on your results
filtering_results = test_filtering_performance(best_collection)


Waiting for collection 'my_domain_balanced' to be indexed...
 - Status: yellow, Points: 1200, Indexed: 1200
 - Status: green, Points: 1200, Indexed: 1200
✅ Collection 'my_domain_balanced' is ready.


In [13]:
print("=" * 60)
print("PERFORMANCE OPTIMIZATION RESULTS")
print("=" * 60)

print("\n1) Upload Performance:")
for config_name, time_taken in upload_times.items():
    print(f"   {config_name}: {time_taken:.2f}s")

print("\n2) Search Performance (hnsw_ef=128):")
for config_name, results in performance_results.items():
    if 128 in results:
        print(f"   {config_name}: {results[128]['avg_time']:.2f}ms")

print("\n3) Filtering Impact:")
print(f"   Without index: {filtering_results['without_index']:.2f}ms")
print(f"   With index: {filtering_results['with_index']:.2f}ms")
print(f"   Speedup: {filtering_results['speedup']:.1f}x")

PERFORMANCE OPTIMIZATION RESULTS

1) Upload Performance:
   fast_initial_upload: 3.44s
   memory_optimized: 3.65s
   balanced: 3.53s
   high_quality: 3.62s

2) Search Performance (hnsw_ef=128):
   memory_optimized: 111.04ms
   balanced: 110.71ms
   high_quality: 112.18ms

3) Filtering Impact:
   Without index: 115.12ms
   With index: 110.98ms
   Speedup: 1.0x
